# GOLD TRANSFORMATION

New good gold

In [0]:
# ==============================================================================
# GOLD - CELL 1 of 3: shared setup + DIMENSION tables
# Run order: this cell -> facts cell -> KPI cell
# (facts need the dimension keys, so the dimensions come first)
# ==============================================================================

# ---------------------------------------------------------------- 1. Table names
# Change a table name here and the whole Gold layer follows.
BASE = f"{CATALOG_NAME}.{SCHEMA_NAME}"

SILVER_EQUIPMENT = f"{BASE}.test_silver_equipment"
SILVER_WORK_ORDERS = f"{BASE}.test_silver_work_orders"
SILVER_MAINTENANCE_HISTORY = f"{BASE}.test_silver_maintenance_history"
SILVER_SENSOR_EVENTS = f"{BASE}.test_silver_sensor_events"

DIM_DATE = f"{BASE}.test_gold_dim_date"
DIM_SITE = f"{BASE}.test_gold_dim_site"
DIM_EQUIPMENT_TYPE = f"{BASE}.test_gold_dim_equipment_type"
DIM_EQUIPMENT = f"{BASE}.test_gold_dim_equipment"
DIM_FAILURE_MODE = f"{BASE}.test_gold_dim_failure_mode"
DIM_MAINTENANCE_TYPE = f"{BASE}.test_gold_dim_maintenance_type"

FACT_SENSOR_READING = f"{BASE}.test_gold_fact_sensor_reading"
FACT_WORK_ORDER = f"{BASE}.test_gold_fact_maintenance_work_order"
FACT_DOWNTIME = f"{BASE}.test_gold_fact_downtime"

KPI_TABLE = f"{BASE}.test_gold_kpi_equipment_month"
KPI_VIEW = f"{BASE}.test_gold_vw_kpi_equipment_month_v2"

# ---------------------------------------------------------------- 2. Settings
UNKNOWN_KEY = -1                 # key of the "Unknown" row in every dimension
AS_OF_OVERRIDE = None            # e.g. "2026-09-30 23:59:59"; None = latest timestamp in Silver
CLOSED_STATUSES = ["Closed", "Cancelled"]
CANCELLED_STATUSES = ["Cancelled"]  # cancelled work orders: kept as rows, excluded from counts and hours_to_close
UNPLANNED_TYPES = ["Corrective"]  # only Corrective work orders are unplanned
ALARM_STATES = ["warning", "alarm"]
TEMP_FIX = "Temporary fix"
REPEAT_DAYS = 30                 # repeat failure = same asset + failure mode within 30 days
TS_FORMAT = "yyyy-MM-dd HH:mm:ss"

FAILURE_MODES = {
    "NONE": "No failure (planned work)",
    "BRG-WEAR": "Bearing wear",
    "MISALIGN": "Shaft misalignment",
    "IMBAL": "Rotor imbalance",
    "OVERHEAT": "Overheating",
    "LUB-FAIL": "Lubrication failure",
    "WIND-INS": "Stator winding insulation breakdown",
    "SEAL-LEAK": "Mechanical seal leakage",
    "CAVIT": "Cavitation",
    "VALVE-FAIL": "Valve failure",
    "GEAR-WEAR": "Gear tooth wear",
    "CLOG": "Blocked suction / plugged strainer",
}
MAINTENANCE_TYPES = ["Preventive", "Corrective", "Inspection", "Predictive"]

spark.conf.set("spark.sql.session.timeZone", "UTC")


# ---------------------------------------------------------------- 3. Small helpers
# (the facts and KPI cells reuse these)
def clean_code(column):
    """Trim, upper-case, empty -> NULL. Used for equipment IDs and site codes."""
    return F.nullif(F.upper(F.trim(F.col(column))), F.lit(""))


def make_key(column):
    """Surrogate key: a hash of the business key. Same input = same key on every run."""
    return F.abs(F.xxhash64(column))


def key_or_unknown(column):
    return F.when(column.isNull(), F.lit(UNKNOWN_KEY)).otherwise(make_key(column))


def date_key(column):
    """Timestamp or date -> 20260315. NULL -> -1 (Unknown)."""
    return F.when(column.isNull(), F.lit(UNKNOWN_KEY)).otherwise(
        F.date_format(column, "yyyyMMdd").cast("int")
    )


def hours_between(start, end):
    return (F.unix_timestamp(end) - F.unix_timestamp(start)) / 3600.0


def add_unknown_row(df, **values):
    """Add the Unknown member (key -1). Columns that are not given stay NULL."""
    row = spark.range(1).select(
        *[F.lit(values.get(f.name)).cast(f.dataType).alias(f.name) for f in df.schema.fields]
    )
    return df.unionByName(row)


def save_table(df, table):
    """Overwrite the Gold table. overwriteSchema is on, so new or changed columns replace the old
    schema without dropping the table. Schema changes no longer fail loudly: rely on the checks."""
    df = df.withColumn("gold_loaded_ts", F.current_timestamp())  # last-refresh indicator
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"{table}: {spark.table(table).count()} rows")


FAILED_CHECKS = []


def check(name, expected, actual, tolerance=0):
    """Print PASS / FAIL for one reconciliation check."""
    expected, actual = float(expected or 0), float(actual or 0)
    ok = abs(actual - expected) <= tolerance
    print(f"{'PASS' if ok else 'FAIL'} | {name} | expected={expected:,.2f} actual={actual:,.2f}")
    if not ok:
        FAILED_CHECKS.append(name)


FAILED_CHECKS.clear()

# ---------------------------------------------------------------- 4. Read Silver
silver_equipment = spark.table(SILVER_EQUIPMENT)
silver_wo = spark.table(SILVER_WORK_ORDERS)
silver_sensor = spark.table(SILVER_SENSOR_EVENTS)

# As-of timestamp = latest event in Silver. Used later for overdue work orders and
# downtime that has not ended yet. Fixed here so every Gold table uses the same value.
wo_event_cols = ["created_ts", "actual_start_ts", "closed_ts", "downtime_start_ts", "downtime_end_ts"]
latest_sensor = silver_sensor.agg(F.date_format(F.max("event_timestamp"), TS_FORMAT)).first()[0]
latest_wo = silver_wo.agg(F.date_format(F.max(F.greatest(*wo_event_cols)), TS_FORMAT)).first()[0]
AS_OF = AS_OF_OVERRIDE or max(t for t in [latest_sensor, latest_wo] if t)
print("As-of timestamp:", AS_OF)

# Date range of dim_date: whole months from the first to the last date in the data
# (due_ts can be in the future, so it is included)
wo_date_cols = wo_event_cols + ["due_ts"]
first_date = min(
    silver_sensor.agg(F.min(F.col("event_timestamp").cast("date"))).first()[0],
    silver_wo.agg(F.min(F.least(*wo_date_cols)).cast("date")).first()[0],
)
last_date = max(
    silver_sensor.agg(F.max(F.col("event_timestamp").cast("date"))).first()[0],
    silver_wo.agg(F.max(F.greatest(*wo_date_cols)).cast("date")).first()[0],
)

# ---------------------------------------------------------------- 5. dim_date
days = spark.range(1).select(
    F.explode(
        F.sequence(F.trunc(F.lit(first_date), "month"), F.last_day(F.lit(last_date)), F.expr("INTERVAL 1 DAY"))
    ).alias("full_date")
)
dim_date = days.select(
    date_key(F.col("full_date")).alias("date_key"),
    "full_date",
    F.year("full_date").alias("year"),
    F.quarter("full_date").alias("quarter"),
    F.month("full_date").alias("month_number"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.date_format("full_date", "yyyy-MM").alias("year_month"),
    F.trunc("full_date", "month").alias("month_start_date"),  # the KPI table joins on this
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.dayofweek("full_date").isin(1, 7).alias("is_weekend"),
)
dim_date = add_unknown_row(
    dim_date, date_key=UNKNOWN_KEY, month_name="Unknown", year_month="UNKNOWN", day_name="Unknown"
)

# ---------------------------------------------------------------- 6. Current equipment
# Equipment IDs differ in casing between Silver tables, so every table uses upper(trim()).
equipment = (
    silver_equipment
    .withColumn("equipment_id", clean_code("equipment_id"))
    .withColumn("site_code", clean_code("site_code"))
    .withColumn("equipment_type", F.nullif(F.trim("equipment_type"), F.lit("")))
    .filter(F.col("equipment_id").isNotNull())
)
# keep one row per equipment: the latest version (this makes dim_equipment SCD type 1)
latest_first = Window.partitionBy("equipment_id").orderBy(
    F.col("record_version").desc(), F.col("last_modified_ts").desc()
)
equipment = equipment.withColumn("rn", F.row_number().over(latest_first)).filter("rn = 1").drop("rn")

# ---------------------------------------------------------------- 7. dim_site
dim_site = (
    equipment.filter(F.col("site_code").isNotNull())
    .groupBy("site_code")
    .agg(
        F.max("site_name").alias("site_name"),
        F.max("country").alias("country"),
        F.max("site_timezone").alias("site_timezone"),
    )
    .withColumn("site_key", make_key(F.col("site_code")))
    .select("site_key", "site_code", "site_name", "country", "site_timezone")
)
dim_site = add_unknown_row(dim_site, site_key=UNKNOWN_KEY, site_code="UNKNOWN", site_name="Unknown site")

# ---------------------------------------------------------------- 8. dim_equipment_type
dim_equipment_type = (
    equipment.filter(F.col("equipment_type").isNotNull())
    .select("equipment_type").distinct()
    .withColumn("equipment_type_key", make_key(F.col("equipment_type")))
    .select("equipment_type_key", "equipment_type")
)
dim_equipment_type = add_unknown_row(
    dim_equipment_type, equipment_type_key=UNKNOWN_KEY, equipment_type="UNKNOWN"
)

# ---------------------------------------------------------------- 9. dim_equipment
# operating_hours_per_day / operating_days_per_week are carried so the KPI cell can build operating_share
dim_equipment = equipment.select(
    make_key(F.col("equipment_id")).alias("equipment_key"),
    "equipment_id", "equipment_name", "manufacturer", "model_number", "production_line",
    "criticality", "status", "install_date", "decommission_date", "rated_power_kw",
    "operating_hours_per_day", "operating_days_per_week",
    key_or_unknown(F.col("site_code")).alias("site_key"),
    key_or_unknown(F.col("equipment_type")).alias("equipment_type_key"),
)
dim_equipment = add_unknown_row(
    dim_equipment, equipment_key=UNKNOWN_KEY, equipment_id="UNKNOWN", equipment_name="Unknown equipment",
    site_key=UNKNOWN_KEY, equipment_type_key=UNKNOWN_KEY,
)

# ---------------------------------------------------------------- 10. dim_failure_mode
dim_failure_mode = spark.createDataFrame(
    list(FAILURE_MODES.items()), "failure_mode_code STRING, failure_mode_desc STRING"
).withColumn("failure_mode_key", make_key(F.col("failure_mode_code")))
dim_failure_mode = dim_failure_mode.select("failure_mode_key", "failure_mode_code", "failure_mode_desc")
dim_failure_mode = add_unknown_row(
    dim_failure_mode, failure_mode_key=UNKNOWN_KEY, failure_mode_code="UNKNOWN",
    failure_mode_desc="Unknown failure mode",
)

# ---------------------------------------------------------------- 11. dim_maintenance_type
# is_planned is False only for Corrective work (Predictive is condition-triggered but scheduled)
dim_maintenance_type = spark.createDataFrame(
    [(t, t not in UNPLANNED_TYPES) for t in MAINTENANCE_TYPES],
    "maintenance_type STRING, is_planned BOOLEAN",
).withColumn("maintenance_type_key", make_key(F.col("maintenance_type")))
dim_maintenance_type = dim_maintenance_type.select("maintenance_type_key", "maintenance_type", "is_planned")
dim_maintenance_type = add_unknown_row(
    dim_maintenance_type, maintenance_type_key=UNKNOWN_KEY, maintenance_type="UNKNOWN"
)

# ---------------------------------------------------------------- 12. Save
save_table(dim_date, DIM_DATE)
save_table(dim_site, DIM_SITE)
save_table(dim_equipment_type, DIM_EQUIPMENT_TYPE)
save_table(dim_equipment, DIM_EQUIPMENT)
save_table(dim_failure_mode, DIM_FAILURE_MODE)
save_table(dim_maintenance_type, DIM_MAINTENANCE_TYPE)

# ---------------------------------------------------------------- 13. Checks
# Every surrogate key must be unique (a hash collision would show up here)
for table, key in [
    (DIM_DATE, "date_key"), (DIM_SITE, "site_key"), (DIM_EQUIPMENT_TYPE, "equipment_type_key"),
    (DIM_EQUIPMENT, "equipment_key"), (DIM_FAILURE_MODE, "failure_mode_key"),
    (DIM_MAINTENANCE_TYPE, "maintenance_type_key"),
]:
    df = spark.table(table)
    check(f"{key} is unique in {table}", df.count(), df.select(key).distinct().count())

silver_equipment_count = silver_equipment.select(clean_code("equipment_id")).dropna().distinct().count()
check("dim_equipment = distinct Silver equipment + Unknown row",
      silver_equipment_count + 1, spark.table(DIM_EQUIPMENT).count())

assert not FAILED_CHECKS, f"Dimension checks failed: {FAILED_CHECKS}"

As-of timestamp: 2026-09-21 23:49:34
data2026acc123456.team3.test_gold_dim_date: 246 rows
data2026acc123456.team3.test_gold_dim_site: 3 rows
data2026acc123456.team3.test_gold_dim_equipment_type: 10 rows
data2026acc123456.team3.test_gold_dim_equipment: 9590 rows
data2026acc123456.team3.test_gold_dim_failure_mode: 13 rows
data2026acc123456.team3.test_gold_dim_maintenance_type: 5 rows
PASS | date_key is unique in data2026acc123456.team3.test_gold_dim_date | expected=246.00 actual=246.00
PASS | site_key is unique in data2026acc123456.team3.test_gold_dim_site | expected=3.00 actual=3.00
PASS | equipment_type_key is unique in data2026acc123456.team3.test_gold_dim_equipment_type | expected=10.00 actual=10.00
PASS | equipment_key is unique in data2026acc123456.team3.test_gold_dim_equipment | expected=9,590.00 actual=9,590.00
PASS | failure_mode_key is unique in data2026acc123456.team3.test_gold_dim_failure_mode | expected=13.00 actual=13.00
PASS | maintenance_type_key is unique in data2026acc1

In [0]:
# ==============================================================================
# GOLD - CELL 2 of 3: FACT tables
# Run after the dimensions cell (it uses its table names, settings and helpers).
#   fact_sensor_reading          one reading per equipment, sensor and timestamp
#   fact_maintenance_work_order  one work order
#   fact_downtime                one non-overlapping downtime interval per equipment
# ==============================================================================
FAILED_CHECKS.clear()

# ---------------------------------------------------------------- 1. Read Silver and dimensions
silver_sensor = spark.table(SILVER_SENSOR_EVENTS)
silver_wo = spark.table(SILVER_WORK_ORDERS)
silver_history = spark.table(SILVER_MAINTENANCE_HISTORY)

dim_equipment = spark.table(DIM_EQUIPMENT).select("equipment_id", "equipment_key", "site_key", "equipment_type_key")
dim_failure_mode = spark.table(DIM_FAILURE_MODE).select("failure_mode_code", "failure_mode_key")
dim_maintenance_type = spark.table(DIM_MAINTENANCE_TYPE).select(
    F.col("maintenance_type").alias("wo_type"), "maintenance_type_key"
)

EQUIPMENT_KEYS = ["equipment_key", "site_key", "equipment_type_key"]


def add_keys(df, dim, join_column, key_columns):
    """Look up the dimension keys. No match (or NULL) -> -1, the Unknown member."""
    return df.join(dim, join_column, "left").fillna(UNKNOWN_KEY, subset=key_columns)


as_of = F.lit(AS_OF).cast("timestamp")

# ---------------------------------------------------------------- 2. fact_sensor_reading
sensor = silver_sensor.withColumn("equipment_id", clean_code("equipment_id"))

# Silver removes exact duplicates only. Here we keep ONE row per (equipment, sensor, timestamp):
# the one with the highest record_version.
one_per_reading = Window.partitionBy(
    F.coalesce("equipment_id", F.lit("")), "sensor_id", "event_timestamp"
).orderBy(F.col("record_version").desc(), F.col("event_id"))
sensor = sensor.withColumn("rn", F.row_number().over(one_per_reading)).filter("rn = 1")

sensor = add_keys(sensor, dim_equipment, "equipment_id", EQUIPMENT_KEYS)

fact_sensor_reading = sensor.select(
    date_key(F.col("event_timestamp")).alias("date_key"),
    "equipment_key", "site_key", "equipment_type_key",
    F.col("event_timestamp").alias("event_ts"),
    "equipment_id", "sensor_id", "sensor_type", "event_id", "operating_state",
    F.col("reading_value_canonical").alias("reading_value"),   # already in SI units
    "canonical_unit", "alarm_state", "alarm_code",
    F.coalesce(F.col("alarm_state").isin(ALARM_STATES), F.lit(False)).alias("is_alarm"),
    F.col("_batch_id").alias("batch_id"),
    F.col("_source_file").alias("source_file"),
)

# ---------------------------------------------------------------- 3. fact_maintenance_work_order
# From the maintenance history we take the activity count, the temporary-fix flag and the activity parts cost.
# The history cost is used only to recover a missing work order parts cost (the work order cost is the source).
history = (
    silver_history.filter(F.col("work_order_id").isNotNull())
    .groupBy("work_order_id")
    .agg(
        F.count(F.lit(1)).alias("activity_count"),
        F.max((F.col("outcome") == TEMP_FIX).cast("int")).alias("temp_fix"),
        F.sum("parts_cost_eur").alias("activity_parts_cost_eur"),
    )
)

wo = (
    silver_wo
    .withColumn("equipment_id", clean_code("equipment_id"))
    .join(history, "work_order_id", "left")
)
wo = add_keys(wo, dim_equipment, "equipment_id", EQUIPMENT_KEYS)
wo = add_keys(wo, dim_failure_mode, "failure_mode_code", ["failure_mode_key"])
wo = add_keys(wo, dim_maintenance_type, "wo_type", ["maintenance_type_key"])

is_cancelled = F.coalesce(F.col("status").isin(CANCELLED_STATUSES), F.lit(False))

# Closed work order with no parts cost (Silver nulled a negative value): recover from activities.
needs_recovery = F.coalesce(
    (F.col("status") == "Closed")
    & F.col("parts_cost_eur").isNull()
    & F.col("activity_parts_cost_eur").isNotNull(),
    F.lit(False),
)
wo = (
    wo.withColumn("parts_cost_recovered", needs_recovery)
      .withColumn(
          "parts_cost_eur",
          F.when(F.col("parts_cost_recovered"), F.col("activity_parts_cost_eur"))
           .otherwise(F.col("parts_cost_eur"))
           .cast("decimal(12,2)"),
      )
)

is_open = ~F.col("status").isin(CLOSED_STATUSES)

fact_work_order = wo.select(
    "work_order_id",
    date_key(F.col("created_ts")).alias("created_date_key"),
    date_key(F.col("due_ts")).alias("due_date_key"),
    date_key(F.col("closed_ts")).alias("closed_date_key"),
    "equipment_key", "site_key", "equipment_type_key", "failure_mode_key", "maintenance_type_key",
    "equipment_id", "priority", "status", "root_cause_code",
    "created_ts", "due_ts", "actual_start_ts", "closed_ts",
    "labor_hours", "labor_cost_eur", "parts_cost_eur",
    (F.coalesce("labor_cost_eur", F.lit(0)) + F.coalesce("parts_cost_eur", F.lit(0))).alias("maintenance_cost_eur"),
    F.when(~is_cancelled, hours_between(F.col("created_ts"), F.col("closed_ts"))).alias("hours_to_close"),
    is_open.alias("is_open"),
    # overdue = still open and the due date has passed
    F.coalesce(is_open & (F.col("due_ts") < as_of), F.lit(False)).alias("is_overdue"),
    F.col("parts_cost_recovered"),
    is_cancelled.alias("is_cancelled"),
    F.coalesce((F.col("status") == "Closed") & (F.col("closed_ts") > F.col("due_ts")),
               F.lit(False)).alias("is_late_closed"),
    F.coalesce("activity_count", F.lit(0)).alias("activity_count"),
    (F.coalesce("temp_fix", F.lit(0)) == 1).alias("has_temporary_fix"),
    F.col("_batch_id").alias("batch_id"),
    F.col("_source_file").alias("source_file"),
)

# ---------------------------------------------------------------- 4. fact_downtime
# Source: the downtime columns of the work orders.
downtime = (
    silver_wo.filter(F.col("downtime_start_ts").isNotNull() & F.col("downtime_type").isNotNull())
    .withColumn("equipment_id", clean_code("equipment_id"))
    .withColumn("is_ongoing", F.col("downtime_end_ts").isNull())
    # downtime that has not ended yet is counted until the as-of timestamp
    .withColumn("end_ts", F.coalesce("downtime_end_ts", as_of))
    .filter(F.col("end_ts") >= F.col("downtime_start_ts"))
    # rows without equipment are never compared with each other
    .withColumn("asset", F.coalesce("equipment_id", F.concat(F.lit("UNKNOWN-"), F.col("work_order_id"))))
    .withColumn("failure_mode", F.coalesce("failure_mode_code", F.lit("NONE")))
    .withColumn("is_unplanned", F.col("downtime_type") == "unplanned")
)

# Overlap rule: the interval that starts first wins. A later interval of the same asset
# starts only when the earlier ones have ended (0 hours if it lies completely inside).
earlier_rows = (
    Window.partitionBy("asset")
    .orderBy("downtime_start_ts", "end_ts", "work_order_id")
    .rowsBetween(Window.unboundedPreceding, -1)
)
downtime = downtime.withColumn("latest_earlier_end", F.max("end_ts").over(earlier_rows))
downtime = downtime.withColumn(
    "start_clipped", F.greatest("downtime_start_ts", "latest_earlier_end")  # greatest ignores NULL
)
downtime = downtime.withColumn("end_clipped", F.greatest("end_ts", "start_clipped"))

# Repeat failure: unplanned, and the same asset + failure mode had an unplanned failure
# that ended less than 30 days before this one started.
earlier_same_mode = (
    Window.partitionBy("asset", "failure_mode")
    .orderBy("downtime_start_ts", "end_ts", "work_order_id")
    .rowsBetween(Window.unboundedPreceding, -1)
)
downtime = downtime.withColumn(
    "earlier_failure_end", F.max(F.when(F.col("is_unplanned"), F.col("end_ts"))).over(earlier_same_mode)
)
is_repeat = (
    F.col("is_unplanned")
    & (F.col("failure_mode") != "NONE")
    & F.col("earlier_failure_end").isNotNull()
    & (F.col("downtime_start_ts") <= F.col("earlier_failure_end") + F.expr(f"INTERVAL {REPEAT_DAYS} DAYS"))
)

downtime = add_keys(downtime, dim_equipment, "equipment_id", EQUIPMENT_KEYS)
downtime = add_keys(downtime, dim_failure_mode, "failure_mode_code", ["failure_mode_key"])
downtime = add_keys(downtime, dim_maintenance_type, "wo_type", ["maintenance_type_key"])

fact_downtime = downtime.select(
    "work_order_id",
    date_key(F.col("start_clipped")).alias("date_key"),
    "equipment_key", "site_key", "equipment_type_key", "failure_mode_key", "maintenance_type_key",
    "equipment_id", "downtime_type",
    F.col("start_clipped").alias("downtime_start_ts"),
    F.col("end_clipped").alias("downtime_end_ts"),
    hours_between(F.col("start_clipped"), F.col("end_clipped")).alias("downtime_hours"),
    F.col("downtime_start_ts").alias("source_start_ts"),   # as delivered, before clipping
    hours_between(F.col("downtime_start_ts"), F.col("end_ts")).alias("original_hours"),
    "is_ongoing",
    F.coalesce(is_repeat, F.lit(False)).alias("is_repeat_failure"),
    F.col("_batch_id").alias("batch_id"),
    F.col("_source_file").alias("source_file"),
)

# ---------------------------------------------------------------- 5. Save
save_table(fact_sensor_reading, FACT_SENSOR_READING)
save_table(fact_work_order, FACT_WORK_ORDER)
save_table(fact_downtime, FACT_DOWNTIME)

# Layout: the sensor fact is the only large table and is filtered by asset and date,
# so we co-locate the data on those columns. The other tables are small: no tuning needed.
spark.sql(f"OPTIMIZE {FACT_SENSOR_READING} ZORDER BY (equipment_key, date_key)")

# ---------------------------------------------------------------- 6. Checks (Silver vs Gold)
sensor_f = spark.table(FACT_SENSOR_READING)
wo_f = spark.table(FACT_WORK_ORDER)
dt_f = spark.table(FACT_DOWNTIME)

# row counts
rows_in_grain = (
    silver_sensor.select(F.coalesce(clean_code("equipment_id"), F.lit("")), "sensor_id", "event_timestamp")
    .distinct().count()
)
check("sensor rows = distinct (equipment, sensor, timestamp) in Silver", rows_in_grain, sensor_f.count())
check("work order rows", silver_wo.count(), wo_f.count())
check("work_order_id is unique", wo_f.count(), wo_f.select("work_order_id").distinct().count())

valid_downtime = silver_wo.filter(
    F.col("downtime_start_ts").isNotNull() & F.col("downtime_type").isNotNull()
    & (F.coalesce("downtime_end_ts", as_of) >= F.col("downtime_start_ts"))
)
check("downtime rows", valid_downtime.count(), dt_f.count())

# totals
silver_hours = valid_downtime.agg(
    F.sum(hours_between(F.col("downtime_start_ts"), F.coalesce("downtime_end_ts", as_of)))
).first()[0]
check("downtime hours before clipping", silver_hours, dt_f.agg(F.sum("original_hours")).first()[0], 0.000001)

silver_cost = silver_wo.agg(
    F.sum(F.coalesce("labor_cost_eur", F.lit(0)) + F.coalesce("parts_cost_eur", F.lit(0)))
).first()[0]
recovered = wo_f.filter("parts_cost_recovered").agg(F.sum("parts_cost_eur")).first()[0]
check("maintenance cost EUR (Silver + recovered)",
      float(silver_cost or 0) + float(recovered or 0),
      wo_f.agg(F.sum("maintenance_cost_eur")).first()[0], 0.01)

# no overlapping downtime for the same asset
earlier_in_gold = (
    Window.partitionBy("equipment_key")
    .orderBy("downtime_start_ts", "downtime_end_ts", "work_order_id")
    .rowsBetween(Window.unboundedPreceding, -1)
)
overlaps = (
    dt_f.filter(F.col("equipment_key") != UNKNOWN_KEY)
    .withColumn("latest_earlier_end", F.max("downtime_end_ts").over(earlier_in_gold))
    .filter(F.col("downtime_start_ts") < F.col("latest_earlier_end"))
    .count()
)
check("overlapping downtime intervals", 0, overlaps)

# every key in a fact must exist in its dimension
for fact_table, key, dim_table, dim_key in [
    (FACT_SENSOR_READING, "date_key", DIM_DATE, "date_key"),
    (FACT_SENSOR_READING, "equipment_key", DIM_EQUIPMENT, "equipment_key"),
    (FACT_SENSOR_READING, "site_key", DIM_SITE, "site_key"),
    (FACT_SENSOR_READING, "equipment_type_key", DIM_EQUIPMENT_TYPE, "equipment_type_key"),
    (FACT_WORK_ORDER, "created_date_key", DIM_DATE, "date_key"),
    (FACT_WORK_ORDER, "due_date_key", DIM_DATE, "date_key"),
    (FACT_WORK_ORDER, "closed_date_key", DIM_DATE, "date_key"),
    (FACT_WORK_ORDER, "equipment_key", DIM_EQUIPMENT, "equipment_key"),
    (FACT_WORK_ORDER, "failure_mode_key", DIM_FAILURE_MODE, "failure_mode_key"),
    (FACT_WORK_ORDER, "maintenance_type_key", DIM_MAINTENANCE_TYPE, "maintenance_type_key"),
    (FACT_DOWNTIME, "date_key", DIM_DATE, "date_key"),
    (FACT_DOWNTIME, "equipment_key", DIM_EQUIPMENT, "equipment_key"),
    (FACT_DOWNTIME, "failure_mode_key", DIM_FAILURE_MODE, "failure_mode_key"),
    (FACT_DOWNTIME, "maintenance_type_key", DIM_MAINTENANCE_TYPE, "maintenance_type_key"),
]:
    fact, dim = spark.table(fact_table), spark.table(dim_table)
    orphans = fact.join(dim, fact[key] == dim[dim_key], "left_anti").count()
    check(f"keys of {key} found in dimension ({fact_table.split('.')[-1]})", 0, orphans)

# information for the demo (not pass / fail)
print("\nInformation:")
print("  sensor rows removed to keep the grain:", silver_sensor.count() - sensor_f.count())
print("  downtime hours removed by overlap clipping:",
      dt_f.agg(F.sum(F.col("original_hours") - F.col("downtime_hours"))).first()[0])
print("  sensor / work order / downtime rows with Unknown equipment:",
      sensor_f.filter(F.col("equipment_key") == UNKNOWN_KEY).count(),
      wo_f.filter(F.col("equipment_key") == UNKNOWN_KEY).count(),
      dt_f.filter(F.col("equipment_key") == UNKNOWN_KEY).count())
print("  parts cost recovered from activities:",
      wo_f.filter("parts_cost_recovered").count(), "work orders, EUR", recovered)

h_cost = (silver_history.filter("work_order_id IS NOT NULL")
          .groupBy("work_order_id").agg(F.sum("parts_cost_eur").alias("h_parts")))
gap = (silver_wo.filter("parts_cost_eur IS NOT NULL").join(h_cost, "work_order_id")
       .agg(F.count("*").alias("n"),
            F.sum((F.abs(F.col("parts_cost_eur") - F.col("h_parts")) >= 0.01).cast("int")).alias("differ"),
            F.sum("parts_cost_eur").alias("wo_eur"),
            F.sum("h_parts").alias("act_eur")).first())
print("  parts cost work order vs activities: compared", gap["n"], "differ", gap["differ"],
      "| work order EUR", gap["wo_eur"], "| activities EUR", gap["act_eur"])

QUARANTINE = f"{BASE}.test_silver_dq_quarantine"
print("\nOpen quarantine items:")
(spark.table(QUARANTINE).filter("resolution_status = 'OPEN'")
 .groupBy("entity", "rule_id", "quarantine_type").count()
 .orderBy("entity", "rule_id").show(50, truncate=False))

assert not FAILED_CHECKS, f"Fact checks failed: {FAILED_CHECKS}"

data2026acc123456.team3.test_gold_fact_sensor_reading: 9742 rows
data2026acc123456.team3.test_gold_fact_maintenance_work_order: 9584 rows
data2026acc123456.team3.test_gold_fact_downtime: 5397 rows
PASS | sensor rows = distinct (equipment, sensor, timestamp) in Silver | expected=9,742.00 actual=9,742.00
PASS | work order rows | expected=9,584.00 actual=9,584.00
PASS | work_order_id is unique | expected=9,584.00 actual=9,584.00
PASS | downtime rows | expected=5,397.00 actual=5,397.00
PASS | downtime hours before clipping | expected=26,804.47 actual=26,804.47
PASS | maintenance cost EUR (Silver + recovered) | expected=5,538,431.75 actual=5,538,431.75
PASS | overlapping downtime intervals | expected=0.00 actual=0.00
PASS | keys of date_key found in dimension (test_gold_fact_sensor_reading) | expected=0.00 actual=0.00
PASS | keys of equipment_key found in dimension (test_gold_fact_sensor_reading) | expected=0.00 actual=0.00
PASS | keys of site_key found in dimension (test_gold_fact_sensor_r

In [0]:
# ==============================================================================
# GOLD - CELL 3 of 3: KPIs
# Run after the facts cell.
# Step 1: table kpi_equipment_month = the ADDITIVE pieces of every KPI per equipment and month
#         (hours, counts, cost). Pieces can be added up to any level (site, month, fleet).
# Step 2: view vw_kpi_equipment_month = the KPI formulas on top of those pieces.
# ==============================================================================
FAILED_CHECKS.clear()

# Σημείωση: Υποθέτουμε ότι οι μεταβλητές KPI_TABLE, KPI_VIEW_V2, UNKNOWN_KEY κλπ. 
# έχουν ήδη οριστεί στα προηγούμενα κελιά (όπως το είχαμε τρέξει επιτυχώς).

# ---------------------------------------------------------------- 1. KPI formulas
# Εδώ ακριβώς βρίσκεται η λογική που έφτιαξες σε DAX (διαιρέσεις και NULLIF για αποφυγή divide-by-zero)
KPI_FORMULAS = {
    "availability": "(SUM(calendar_hours) - SUM(downtime_hours)) / NULLIF(SUM(calendar_hours), 0)",
    "unplanned_downtime_hours": "SUM(unplanned_downtime_hours)",
    "unplanned_failures": "SUM(unplanned_failure_count)",
    "mtbf_hours": "SUM(operating_uptime_hours) / NULLIF(SUM(unplanned_failure_count), 0)",
    "mttr_hours": "SUM(unplanned_downtime_hours) / NULLIF(SUM(unplanned_failure_count), 0)",
    "overdue_work_orders": "SUM(overdue_work_order_count)",
    "alarm_rate": "SUM(alarm_count) / NULLIF(SUM(asset_days), 0)",
    "alarm_warning_rate": "SUM(alarm_warning_count) / NULLIF(SUM(asset_days), 0)",
    "alarm_critical_rate": "SUM(alarm_critical_count) / NULLIF(SUM(asset_days), 0)",
    "maintenance_cost_eur": "SUM(maintenance_cost_eur)",
    "repeat_failure_rate": "SUM(repeat_failure_count) / NULLIF(SUM(unplanned_failure_count), 0)",
    "work_orders": "SUM(work_order_count)",
}

def kpi_query(group_columns):
    """SQL that calculates all KPIs, grouped by the given columns (empty list = whole fleet)."""
    kpis = ",\n    ".join(f"{formula} AS {name}" for name, formula in KPI_FORMULAS.items())
    if not group_columns:
        return f"SELECT\n    {kpis}\nFROM {KPI_TABLE}"
    groups = ", ".join(group_columns)
    return f"SELECT {groups},\n    {kpis}\nFROM {KPI_TABLE}\nGROUP BY {groups}"

# ---------------------------------------------------------------- 2. Read Gold tables
equipment = spark.table(DIM_EQUIPMENT).filter(F.col("equipment_key") != UNKNOWN_KEY)

# Δυναμική ανάκτηση πεδίων λειτουργίας (operating_hours) απευθείας από τον Silver αν λείπουν
if "operating_hours_per_day" not in equipment.columns:
    silver_eq = spark.table(SILVER_EQUIPMENT).select(
        F.upper(F.trim(F.col("equipment_id"))).alias("equipment_id"),
        "operating_hours_per_day",
        "operating_days_per_week"
    ).dropDuplicates(["equipment_id"])
    equipment = equipment.join(silver_eq, on="equipment_id", how="left")

# Υπολογισμός operating share (ποσοστό λειτουργίας βάσει ωραρίου)
equipment = equipment.withColumn(
    "operating_share",
    (F.coalesce(F.col("operating_hours_per_day").cast("double"), F.lit(24.0)) / 24.0) * 
    (F.coalesce(F.col("operating_days_per_week").cast("double"), F.lit(7.0)) / 7.0)
)

# Εξαίρεση ορφανών/άγνωστων μηχανημάτων από τα τελικά KPIs
sensor_f = spark.table(FACT_SENSOR_READING).filter(F.col("equipment_key") != UNKNOWN_KEY)
wo_f = spark.table(FACT_WORK_ORDER).filter(F.col("equipment_key") != UNKNOWN_KEY)
dt_f = spark.table(FACT_DOWNTIME).filter(F.col("equipment_key") != UNKNOWN_KEY)

as_of = F.lit(AS_OF).cast("timestamp")
data_start = F.lit(
    min(
        spark.table(FACT_SENSOR_READING).agg(F.date_format(F.min("event_ts"), TS_FORMAT)).first()[0],
        spark.table(FACT_WORK_ORDER).agg(F.date_format(F.min("created_ts"), TS_FORMAT)).first()[0],
    )
).cast("timestamp")

def month_of(timestamp_column):
    """First day of the month of a timestamp, as a date."""
    return F.date_trunc("month", timestamp_column).cast("date")

# ---------------------------------------------------------------- 3. Calendar hours per equipment and month
months = (
    spark.table(DIM_DATE).filter(F.col("month_start_date").isNotNull())
    .select("month_start_date").distinct()
    .filter(F.col("month_start_date") <= as_of.cast("date"))
    .filter(F.add_months("month_start_date", 1) > data_start.cast("date"))
)
month_start = F.col("month_start_date").cast("timestamp")
next_month_start = F.add_months("month_start_date", 1).cast("timestamp")
period_start = F.greatest(month_start, data_start, F.col("install_date").cast("timestamp"))
period_end = F.least(next_month_start, as_of, F.date_add("decommission_date", 1).cast("timestamp"))

calendar = (
    equipment.select("equipment_key", "site_key", "equipment_type_key", "install_date", "decommission_date", "operating_share")
    .crossJoin(months)
    .withColumn("calendar_hours", F.greatest(F.lit(0.0), hours_between(period_start, period_end)))
    .drop("install_date", "decommission_date")
)

# ---------------------------------------------------------------- 4. Downtime hours per month
downtime_by_month = (
    dt_f
    .withColumn(
        "month_start_date",
        F.explode(F.sequence(month_of(F.col("downtime_start_ts")), month_of(F.col("downtime_end_ts")), F.expr("INTERVAL 1 MONTH"))),
    )
    .withColumn(
        "hours_in_month",
        F.greatest(
            F.lit(0.0),
            hours_between(
                F.greatest(F.col("downtime_start_ts"), F.col("month_start_date").cast("timestamp")),
                F.least(F.col("downtime_end_ts"), F.add_months("month_start_date", 1).cast("timestamp")),
            ),
        ),
    )
    .groupBy("equipment_key", "month_start_date")
    .agg(
        F.sum("hours_in_month").alias("downtime_hours"),
        F.sum(F.when(F.col("downtime_type") == "unplanned", F.col("hours_in_month")).otherwise(0.0)).alias("unplanned_downtime_hours"),
    )
)

# ---------------------------------------------------------------- 5. Counts per month
failures_by_month = (
    dt_f.filter(F.col("downtime_type") == "unplanned")
    .groupBy("equipment_key", month_of(F.col("source_start_ts")).alias("month_start_date"))
    .agg(
        F.count(F.lit(1)).alias("unplanned_failure_count"),
        F.sum(F.col("is_repeat_failure").cast("int")).alias("repeat_failure_count"),
    )
)

alarms_by_month = (
    sensor_f
    .groupBy("equipment_key", month_of(F.col("event_ts")).alias("month_start_date"))
    .agg(
        F.sum(F.col("is_alarm").cast("int")).alias("alarm_count"),
        F.sum((F.col("alarm_state") == "warning").cast("int")).alias("alarm_warning_count"),
        F.sum((F.col("alarm_state") == "alarm").cast("int")).alias("alarm_critical_count"),
    )
)

created_by_month = (
    wo_f.groupBy("equipment_key", month_of(F.col("created_ts")).alias("month_start_date"))
    .agg(F.count(F.lit(1)).alias("work_order_count"))
)

overdue_by_month = (
    wo_f.filter(F.col("is_overdue"))
    .groupBy("equipment_key", month_of(F.col("due_ts")).alias("month_start_date"))
    .agg(F.count(F.lit(1)).alias("overdue_work_order_count"))
)

cost_by_month = (
    wo_f.groupBy("equipment_key", month_of(F.coalesce("closed_ts", "created_ts")).alias("month_start_date"))
    .agg(F.sum("maintenance_cost_eur").alias("maintenance_cost_eur"))
)

# ---------------------------------------------------------------- 6. Put the pieces together
pieces = [downtime_by_month, failures_by_month, alarms_by_month,
          created_by_month, overdue_by_month, cost_by_month]

event_months = pieces[0].select("equipment_key", "month_start_date")
for piece in pieces[1:]:
    event_months = event_months.unionByName(piece.select("equipment_key", "month_start_date"))
event_months = event_months.distinct().withColumn("has_events", F.lit(True))

kpi = (
    calendar.join(event_months, ["equipment_key", "month_start_date"], "left")
    .filter((F.col("calendar_hours") > 0) | F.col("has_events").isNotNull())
    .drop("has_events")
)
for piece in pieces:
    kpi = kpi.join(piece, ["equipment_key", "month_start_date"], "left")

kpi_equipment_month = kpi.select(
    "equipment_key", "site_key", "equipment_type_key",
    date_key(F.col("month_start_date")).alias("month_start_date_key"),
    "calendar_hours",
    (F.col("calendar_hours") / 24.0).alias("asset_days"),
    
    # ----------------------------------------------------------------
    # KEY LOGIC: Operating Share implementation for accurate MTBF
    (F.col("calendar_hours") * F.col("operating_share")).alias("operating_hours"),
    F.coalesce("downtime_hours", F.lit(0.0)).alias("downtime_hours"),
    (F.greatest(F.lit(0.0), F.col("calendar_hours") - F.coalesce("downtime_hours", F.lit(0.0))) * F.col("operating_share")).alias("operating_uptime_hours"),
    # ----------------------------------------------------------------
    
    F.coalesce("unplanned_downtime_hours", F.lit(0.0)).alias("unplanned_downtime_hours"),
    F.coalesce("unplanned_failure_count", F.lit(0)).alias("unplanned_failure_count"),
    F.coalesce("repeat_failure_count", F.lit(0)).alias("repeat_failure_count"),
    F.coalesce("alarm_count", F.lit(0)).alias("alarm_count"),
    F.coalesce("alarm_warning_count", F.lit(0)).alias("alarm_warning_count"),
    F.coalesce("alarm_critical_count", F.lit(0)).alias("alarm_critical_count"),
    F.coalesce("work_order_count", F.lit(0)).alias("work_order_count"),
    F.coalesce("overdue_work_order_count", F.lit(0)).alias("overdue_work_order_count"),
    F.coalesce("maintenance_cost_eur", F.lit(0)).alias("maintenance_cost_eur"),
    as_of.alias("as_of_ts"),
)

# ---------------------------------------------------------------- 7. Save table and view
kpi_equipment_month.withColumn("gold_loaded_ts", F.current_timestamp())\
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(KPI_TABLE)
print(f"{KPI_TABLE}: {spark.table(KPI_TABLE).count()} rows")

spark.sql(
    f"CREATE OR REPLACE VIEW {KPI_VIEW} AS\n"
    + kpi_query(["equipment_key", "site_key", "equipment_type_key", "month_start_date_key"])
)
print(f"{KPI_VIEW}: created")

# ---------------------------------------------------------------- 8. Checks (KPI pieces vs facts)
totals = spark.table(KPI_TABLE).agg(
    F.sum("downtime_hours"), F.sum("unplanned_downtime_hours"), F.sum("unplanned_failure_count"),
    F.sum("repeat_failure_count"), F.sum("alarm_count"), F.sum("work_order_count"),
    F.sum("overdue_work_order_count"), F.sum("maintenance_cost_eur"),
).first()

unplanned_dt = dt_f.filter(F.col("downtime_type") == "unplanned")
check("downtime hours", dt_f.agg(F.sum("downtime_hours")).first()[0], totals[0], 0.000001)
check("unplanned downtime hours", unplanned_dt.agg(F.sum("downtime_hours")).first()[0], totals[1], 0.000001)
check("unplanned failures", unplanned_dt.count(), totals[2])
check("repeat failures", dt_f.filter(F.col("is_repeat_failure")).count(), totals[3])
check("alarms", sensor_f.filter(F.col("is_alarm")).count(), totals[4])
check("work orders", wo_f.count(), totals[5])
check("overdue work orders", wo_f.filter(F.col("is_overdue")).count(), totals[6])
check("maintenance cost EUR", wo_f.agg(F.sum("maintenance_cost_eur")).first()[0], totals[7], 0.01)

# ---------------------------------------------------------------- 9. Show the KPIs
print("\nKPIs for the whole fleet:")
spark.sql(kpi_query([])).show(truncate=False)

print("KPIs by site:")
spark.sql(kpi_query(["site_key"])).join(
    spark.table(DIM_SITE).select("site_key", "site_code"), "site_key"
).drop("site_key").show(truncate=False)

assert not FAILED_CHECKS, f"KPI checks failed: {FAILED_CHECKS}"

data2026acc123456.team3.test_gold_kpi_equipment_month: 65249 rows
data2026acc123456.team3.test_gold_vw_kpi_equipment_month_v2: created
PASS | downtime hours | expected=26,054.54 actual=26,054.54
PASS | unplanned downtime hours | expected=9,217.98 actual=9,217.98
PASS | unplanned failures | expected=1,211.00 actual=1,211.00
PASS | repeat failures | expected=35.00 actual=35.00
PASS | alarms | expected=253.00 actual=253.00
PASS | work orders (not cancelled) | expected=9,251.00 actual=9,251.00
PASS | overdue work orders | expected=31.00 actual=31.00
PASS | maintenance cost EUR | expected=5,381,319.74 actual=5,381,319.74
PASS | operating_share within 0..1 | expected=0.00 actual=0.00

+------------+------------+-----------------+--------------+------+-----------+------------------+
|equipment_id|install_date|decommission_date|status        |months|work_orders|downtime_hours    |
+------------+------------+-----------------+--------------+------+-----------+------------------+
|EQ-11340    |2